# SafeCrop · Fairness test: is the CLIP database's lead real? (Google Colab)

The CLIP photo database beat the other models in Step 2, but it had advantages the others didn't: it chose from **7** diseases while the others chose from **33** options, it was shown **labelled example photos**, and its examples came from **the same datasets** as the test photos. This notebook removes those advantages one at a time, from the easiest test to the strongest, so you can see which part of the lead is real.

| Option | What changes | Who is tested |
|---|---|---|
| **1. Same options for everyone** | every model chooses from the same 7 diseases, and from the same 33 options | CLIP zero-shot, CLIP database, Gemini |
| **2. Database filled to 33 options** | the database also gets photos of the look-alike diseases and healthy leaves, so it must choose among them too | CLIP database (and Gemini with its references) |
| **3. Fewer examples** | the database gets only 1, 3, 5 or 10 photos per disease instead of about 29 | CLIP database |
| **4. Examples from another source** | the database may not use any photo from the test photo's own dataset, so matching the photo *style* stops working | CLIP database |
| **5. Gemini gets the same examples** | done in Step 2b; its answers are read from your Drive | Gemini + references + descriptions |

At the end, one table and two charts compare every option.

### Before you run

1. **GPU:** *Runtime → Change runtime type → T4 GPU → Save* (CLIP needs about 3 minutes on a GPU).
2. **Gemini key:** the `GEMINI_API_KEY` Colab secret, as in Step 2. Gemini is used only for the jobs in `GEMINI_JOBS`; each job asks about the **210 L3 images** (one request each), so the default two jobs need **420 requests, one day of the free quota**. Set `GEMINI_JOBS = []` to run only the CLIP parts (no quota).
3. **Your images:** `noisy_images.zip` in your Drive, as before. The Step 2 and Step 2b answers are read from `SafeCrop/results`.
4. **Option 2 photos:** the notebook downloads look-alike and healthy photos for 7 of the 26 extra options from PlantVillage by itself. For the others, add your own photos in Drive (section 3 explains how); the notebook uses whatever it finds.
5. **Runtime → Run all.**

Results go to `SafeCrop/results/fairness` in your Drive.

In [ ]:
# Install anything missing (Colab already has most of it). Safe to run again.
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

def ensure(pkg, min_major=None):
    try:
        v = version(pkg)
        if min_major is not None and int(v.split(".")[0]) < min_major:
            raise PackageNotFoundError(pkg)
    except PackageNotFoundError:
        print("installing", pkg, "...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], check=True)

ensure("google-genai", min_major=1)
ensure("transformers")
ensure("openpyxl")
ensure("scikit-learn")
ensure("scikit-image")
try:
    import cv2
except ImportError:
    ensure("opencv-python-headless")
print("google-genai", version("google-genai"), "| scikit-learn", version("scikit-learn"))

In [ ]:
import os, re, io, csv, json, time, glob, math, random, shutil, zipfile, datetime, subprocess
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display, Image as ShowImage
from tqdm.auto import tqdm

def short(e, n=300):
    s = str(e).replace("\n", " ")
    return s if len(s) <= n else s[:n] + " ..."

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
print("pandas", pd.__version__, "| numpy", np.__version__)

## Settings

You can leave these as they are.

In [ ]:
# ---- where the images are ----
SOURCE = "auto"      # "auto": look in Google Drive, ask for an upload if not found | "drive" | "upload"
DATASET_PATH = ""    # optional exact path to noisy_images.zip
OUT_SUBDIR = "fairness"

# ---- CLIP (the same model and vote as Step 2) ----
CLIP_MODEL = "openai/clip-vit-large-patch14"
CLIP_DATABASE_NEIGHBOURS = 5

# ---- option 2: fill the database to all 33 options ----
EXTRA_PER_OPTION = 30            # photos per extra option, about as many as each of our diseases has
DOWNLOAD_PLANTVILLAGE = True     # fetch photos for 7 look-alike/healthy options (and other-source apple scab) from GitHub
EXTRA_FOLDER = "SafeCrop/extra_references"   # in My Drive: your own photos, one folder per option (section 3)
DUPLICATE_LIMIT = 0.97           # an extra photo this similar (CLIP) to one of our photos counts as a copy and is dropped

# ---- option 3: fewer examples ----
SHOTS = [1, 3, 5, 10]            # photos per disease in the database
DRAWS = 10                       # random picks of those photos; the results are averaged

# ---- option 4: examples from another source ----
OTHER_SOURCE_MIN = 5             # a disease is tested only if it has at least this many photos from another dataset

# ---- Gemini (uses the free daily quota of 500 requests) ----
GEMINI_JOBS = ["7_options", "references_33"]   # also "dictionary_1"; [] = no new Gemini requests
GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_LEVELS = ["L3"]           # 210 requests per job; ["L0", "L1", "L2", "L3"] = 840
GEMINI_PHOTOS_PER_DISEASE = None # None = all photos; e.g. 15 = 15 photos per disease (105 requests at L3)
GEMINI_REQUESTS_PER_MINUTE = 14
SEED = 299

for j in GEMINI_JOBS:
    if j not in ("7_options", "references_33", "dictionary_1"):
        raise ValueError(f'Unknown Gemini job "{j}": use "7_options", "references_33" or "dictionary_1"')

## 1. Get the images

As in Step 2: `noisy_images.zip` from your Drive (or an upload). The Step 2 and Step 2b Gemini answers are read from `SafeCrop/results` in your Drive.

In [ ]:
WORK = "/content/safecrop_inference"
DATA_DIR = os.path.join(WORK, "data")
os.makedirs(DATA_DIR, exist_ok=True)

def mount_drive():
    if os.path.isdir("/content/drive/MyDrive"):
        return True
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Couldn't connect to Google Drive:", short(e, 200))
    return os.path.isdir("/content/drive/MyDrive")

def walk(top, max_depth):
    # os.walk that stops max_depth folders below `top` and skips hidden folders
    top = top.rstrip(os.sep)
    base = top.count(os.sep)
    for root, dirs, files in os.walk(top):
        depth = root.count(os.sep) - base
        dirs[:] = sorted(d for d in dirs if not d.startswith(".")) if depth < max_depth else []
        yield root, files

def is_dataset_folder(root, files):
    return "metadata.csv" in files and os.path.isdir(os.path.join(root, "L0"))

def find_dataset_folder(top):
    for root, files in walk(top, max_depth=6):
        if is_dataset_folder(root, files):
            return root
    return None

def unpack(zip_path, dest):
    with zipfile.ZipFile(zip_path) as z:
        members = [m for m in z.namelist()
                   if not m.startswith("__MACOSX") and not os.path.basename(m.rstrip("/")).startswith("._")]
        z.extractall(dest, members)

def prepare(path):
    # returns the folder that holds metadata.csv and L0..L3, unzipping first if needed
    if os.path.isdir(path):
        return find_dataset_folder(path)
    if not (os.path.isfile(path) and zipfile.is_zipfile(path)):
        return None
    dest = os.path.join(DATA_DIR, re.sub(r"[^\w.-]+", "_", os.path.splitext(os.path.basename(path))[0]))
    if os.path.exists(dest):
        shutil.rmtree(dest)
    print("Unzipping", os.path.basename(path), "...")
    unpack(path, dest)
    for _ in range(3):   # handles a zip inside the zip, e.g. the whole 'safecrop dataset' folder downloaded from Drive
        found = find_dataset_folder(dest)
        if found:
            return found
        inner = [os.path.join(r, f) for r, fs in walk(dest, 6) for f in fs if f.lower().endswith(".zip")]
        inner = [p for p in inner if "noisy" in os.path.basename(p).lower()] or inner
        if not inner:
            break
        for p in inner:
            unpack(p, os.path.splitext(p)[0])
            os.remove(p)
    return None

def find_in_drive():
    found = []
    for root, files in walk("/content/drive/MyDrive", max_depth=4):
        if is_dataset_folder(root, files):
            found.append(root)
        for f in files:
            fl = f.lower()
            if fl.endswith(".zip") and "noisy" in fl and "result" not in fl:
                found.append(os.path.join(root, f))
    return sorted(found, key=os.path.getmtime, reverse=True)

DRIVE_OK, dataset_dir, uploaded = False, None, []

if DATASET_PATH:
    if DATASET_PATH.startswith("/content/drive"):
        DRIVE_OK = mount_drive()
    dataset_dir = prepare(DATASET_PATH)
    if dataset_dir is None:
        print("No images found at DATASET_PATH:", DATASET_PATH)

if dataset_dir is None and SOURCE in ("auto", "drive"):
    DRIVE_OK = mount_drive()
    if DRIVE_OK:
        print("Searching your Google Drive for noisy_images ...")
        candidates = find_in_drive()
        for c in candidates:
            print("  found:", c)
        for c in candidates:
            dataset_dir = prepare(c)
            if dataset_dir:
                print("Using:", c)
                break
        if dataset_dir is None:
            print("No noisy_images.zip found in your Drive. (Files in 'Shared with me' only count if you add a shortcut to My Drive.)")

if dataset_dir is None and SOURCE in ("auto", "upload"):
    from google.colab import files
    print("Click 'Choose files' and pick noisy_images.zip. Without Drive, also add the answer files from "
          "earlier notebooks (gemini_3.5_flash_lite_predictions.csv, ..._help_both_predictions.csv).")
    os.makedirs(os.path.join(WORK, "uploads"), exist_ok=True)
    for name, data in files.upload().items():
        p = os.path.join(WORK, "uploads", name)
        with open(p, "wb") as f:
            f.write(data)
        uploaded.append(p)
    for p in uploaded:
        if p.lower().endswith(".zip"):
            dataset_dir = prepare(p)
            if dataset_dir:
                break

if dataset_dir is None:
    raise RuntimeError("No images found. Put noisy_images.zip in your Google Drive (or set SOURCE = 'upload') and run again.")

# ---- read the list of images ----
meta = pd.read_csv(os.path.join(dataset_dir, "metadata.csv"))
need = {"image", "photo_id", "disease", "level"}
if need - set(meta.columns):
    raise RuntimeError(f"metadata.csv is missing the columns {sorted(need - set(meta.columns))}")
for col in ["disease_name", "group", "origin", "noise"]:
    if col not in meta.columns:
        meta[col] = ""
meta = meta.fillna("")
meta["disease_name"] = [n if n else d for n, d in zip(meta.disease_name, meta.disease)]
meta["level"] = meta.level.astype(str)
meta["path"] = [os.path.join(dataset_dir, p) for p in meta.image]
missing = ~meta.path.map(os.path.exists)
if missing.any():
    print(f"Warning: {missing.sum()} images listed in metadata.csv are missing and will be skipped.")
    meta = meta[~missing]
meta = meta.drop_duplicates("image").sort_values(["disease", "photo_id", "level"]).reset_index(drop=True)

# ---- where results go ----
RESULTS_DIR = "/content/drive/MyDrive/SafeCrop/results" if DRIVE_OK else os.path.join(WORK, "results")
OUT_DIR = os.path.join(RESULTS_DIR, OUT_SUBDIR)       # everything this notebook writes
os.makedirs(OUT_DIR, exist_ok=True)
GEMINI_CACHE = os.path.join(RESULTS_DIR, "gemini_3.5_flash_lite_predictions.csv")   # Step 2: plain Gemini answers
HELP_CACHE = os.path.join(RESULTS_DIR, "gemini_help", "gemini_3.5_flash_lite_help_both_predictions.csv")   # Step 2b

def merge_upload(src, dest):
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    old = pd.read_csv(src, dtype=str, keep_default_na=False)
    if os.path.exists(dest):
        old = pd.concat([pd.read_csv(dest, dtype=str, keep_default_na=False), old])
    old.drop_duplicates(["image", "model"], keep="last").to_csv(dest, index=False)
    print("Using", os.path.basename(src))

for p in uploaded:   # answers from earlier notebooks, when working without Drive
    name = os.path.basename(p).lower()
    if name.endswith(".csv") and "gemini" in name:
        merge_upload(p, HELP_CACHE if "help" in name else GEMINI_CACHE)

print(f"\n{len(meta)} images, {meta.photo_id.nunique()} photos, {meta.disease.nunique()} diseases. Results go to {OUT_DIR}")
display(meta.pivot_table(index=["group", "disease_name"], columns="level", values="image", aggfunc="count", fill_value=0))

## 2. The 33 options and where our photos come from

The 33 options are the same as in Step 2. For option 4, the notebook also needs to know which **dataset** each of our photos came from (from the report's Table 1). Potato late blight and grape leaf blight both come from PlantVillage.

In [ ]:
# key, crop, name shown to Gemini (after "Crop: "), full name used in CLIP's sentences
LABELS = [dict(key=k, crop=c, gemini=g, clip=n) for k, c, g, n in [
    ("apple_scab",                      "apple",      "scab",                                  "apple scab"),
    ("apple_cedar_rust",                "apple",      "cedar apple rust",                      "cedar apple rust"),
    ("apple_alternaria_leaf_spot",      "apple",      "Alternaria leaf spot",                  "apple Alternaria leaf spot"),
    ("apple_powdery_mildew",            "apple",      "powdery mildew",                        "apple powdery mildew"),
    ("apple_healthy",                   "apple",      "healthy",                               "healthy apple"),
    ("potato_late_blight",              "potato",     "late blight",                           "potato late blight"),
    ("potato_early_blight",             "potato",     "early blight",                          "potato early blight"),
    ("potato_healthy",                  "potato",     "healthy",                               "healthy potato"),
    ("grape_leaf_blight",               "grape",      "leaf blight (Isariopsis leaf spot)",    "grape leaf blight"),
    ("grape_black_rot",                 "grape",      "black rot",                             "grape black rot"),
    ("grape_esca",                      "grape",      "esca (black measles)",                  "grape esca"),
    ("grape_downy_mildew",              "grape",      "downy mildew",                          "grape downy mildew"),
    ("grape_healthy",                   "grape",      "healthy",                               "healthy grape"),
    ("pear_leaf_spot",                  "pear",       "leaf spot",                             "pear leaf spot"),
    ("pear_scab",                       "pear",       "scab",                                  "pear scab"),
    ("pear_fire_blight",                "pear",       "fire blight",                           "pear fire blight"),
    ("pear_rust",                       "pear",       "rust",                                  "pear rust"),
    ("pear_healthy",                    "pear",       "healthy",                               "healthy pear"),
    ("rice_stem_borer",                 "rice",       "stem borer damage (dead heart / white head)", "rice stem borer damage"),
    ("rice_blast",                      "rice",       "blast",                                 "rice blast"),
    ("rice_brown_spot",                 "rice",       "brown spot",                            "rice brown spot"),
    ("rice_bacterial_leaf_blight",      "rice",       "bacterial leaf blight",                 "rice bacterial leaf blight"),
    ("rice_sheath_blight",              "rice",       "sheath blight",                         "rice sheath blight"),
    ("rice_healthy",                    "rice",       "healthy",                               "healthy rice"),
    ("jute_mosaic",                     "jute",       "mosaic",                                "jute mosaic"),
    ("jute_dieback",                    "jute",       "dieback",                               "jute dieback"),
    ("jute_stem_rot",                   "jute",       "stem rot",                              "jute stem rot"),
    ("jute_anthracnose",                "jute",       "anthracnose",                           "jute anthracnose"),
    ("jute_healthy",                    "jute",       "healthy",                               "healthy jute"),
    ("black_gram_yellow_mosaic",        "black gram", "yellow mosaic",                         "black gram yellow mosaic"),
    ("black_gram_cercospora_leaf_spot", "black gram", "Cercospora leaf spot",                  "black gram Cercospora leaf spot"),
    ("black_gram_powdery_mildew",       "black gram", "powdery mildew",                        "black gram powdery mildew"),
    ("black_gram_healthy",              "black gram", "healthy",                               "healthy black gram"),
]]

# one line per option: what it looks like on the plant (only the new Gemini predictor sees these)
DESCRIPTIONS = {
    "apple_scab": "olive-green to dark brown velvety spots on the leaves that turn dark and scabby; leaves may pucker",
    "apple_cedar_rust": "bright yellow-orange round spots on the upper leaf, often with a red rim; small tubes or cups underneath",
    "apple_alternaria_leaf_spot": "many small round brown spots with a darker purple-brown rim that can merge into larger dead patches",
    "apple_powdery_mildew": "white powdery coating on leaves and shoot tips; leaves narrow, curled and crinkled",
    "apple_healthy": "evenly green apple leaves with no spots, coating or discoloured patches",
    "potato_late_blight": "large dark brown to black water-soaked patches, often from the leaf tip or edge, with a pale border and white mould underneath in damp weather",
    "potato_early_blight": "brown spots with rings inside like a target, mostly on older lower leaves, often with yellow around them",
    "potato_healthy": "evenly green potato leaves with no spots, blotches or yellowing",
    "grape_leaf_blight": "irregular dark brown to black angular spots that join into large dry dead areas on the leaf",
    "grape_black_rot": "round tan to brown spots with a dark brown rim and tiny black dots inside",
    "grape_esca": "yellow or red-brown stripes between the veins with green along the veins (tiger stripes), parts drying out",
    "grape_downy_mildew": "yellow oily-looking patches on the upper leaf with white downy growth underneath",
    "grape_healthy": "evenly green grape leaves with no spots, stripes or mould",
    "pear_leaf_spot": "many small round grey to brown spots with a dark rim scattered over the leaf",
    "pear_scab": "olive-black velvety spots on leaves and fruit, like apple scab",
    "pear_fire_blight": "leaves and shoot tips turn black or brown as if burnt and stay attached; shoot tip bends like a hook",
    "pear_rust": "bright orange spots on the upper leaf that later form raised bumps or horn-like growths underneath",
    "pear_healthy": "evenly green pear leaves with no spots, scorching or rust",
    "rice_stem_borer": "pest damage: the central shoot dries and turns brown while outer leaves stay green (dead heart), or the grain head stays upright, empty and white (white head)",
    "rice_blast": "diamond or spindle-shaped spots with grey or white centres and brown edges on the leaves",
    "rice_brown_spot": "many small oval brown spots, often with a grey centre and a yellow halo, scattered over the leaves",
    "rice_bacterial_leaf_blight": "yellow to white wavy stripes from the leaf tip and edges; leaves dry out from the tip down",
    "rice_sheath_blight": "oval grey-green to white patches with brown edges on the leaf sheath near the water, like snake skin",
    "rice_healthy": "green upright rice plants with no spots, stripes, dried shoots or white heads",
    "jute_mosaic": "leaves with a patchy yellow and green mosaic pattern, sometimes crinkled or smaller than normal",
    "jute_dieback": "shoots and branches dry out and die from the tip downward; leaves wilt and turn brown or black",
    "jute_stem_rot": "dark brown to black patches on the stem that spread, so the plant wilts or breaks; leaves may get brown spots",
    "jute_anthracnose": "sunken dark brown spots or cracks on the stem and brown spots on the leaves",
    "jute_healthy": "evenly green jute leaves and stems with no spots, mosaic or drying",
    "black_gram_yellow_mosaic": "bright yellow patches mixed with green across the leaves, spreading until leaves turn mostly yellow",
    "black_gram_cercospora_leaf_spot": "small round brown spots with grey centres and red-brown rims on the leaves",
    "black_gram_powdery_mildew": "white powdery patches on the leaves",
    "black_gram_healthy": "evenly green black gram leaves with no spots, yellowing or powder",
}

# a disease folder that isn't in the list above still works: it gets its own option
known = {l["key"] for l in LABELS}
for d, name in meta[["disease", "disease_name"]].drop_duplicates().itertuples(index=False):
    if d not in known:
        crop = name.split()[0].lower() if name else "plant"
        LABELS.append(dict(key=d, crop=crop, gemini=name, clip=name.lower()))
        DESCRIPTIONS.setdefault(d, "")
        print("Added an option for", d)

LABEL_KEYS = [l["key"] for l in LABELS]
LABEL_CROP = {l["key"]: l["crop"] for l in LABELS}
OPTIONS = [f"{l['crop'][0].upper() + l['crop'][1:]}: {l['gemini']}" for l in LABELS]   # what Gemini sees
OPTION_TO_KEY = dict(zip(OPTIONS, LABEL_KEYS))
KEY_TO_OPTION = dict(zip(LABEL_KEYS, OPTIONS))
TARGETS = sorted(meta.disease.unique())


OUR_SOURCE = {"potato_late_blight": "PlantVillage", "grape_leaf_blight": "PlantVillage", "apple_scab": "AppleLeaf9",
              "pear_leaf_spot": "DiaMOS Plant", "rice_stem_borer": "Mendeley rice", "jute_mosaic": "Harvard jute",
              "jute_dieback": "Harvard jute"}
meta["source"] = meta.disease.map(OUR_SOURCE).fillna("our photos")
display(meta.drop_duplicates("photo_id").groupby(["disease_name", "source"]).size().rename("photos").to_frame())

## 2. Helpers

Shared code: the 33 options, photo tools (fixes and quality measurements), CLIP and Gemini helpers. You don't need to change anything here.

In [ ]:
# ===== SafeCrop photo tools: fixes for noisy leaf photos, and quality measurements =====
# Every function takes and returns a 224 x 224 (or any size) RGB photo as a uint8 numpy array.
import numpy as np
import cv2
from PIL import Image


def to_array(img):
    """PIL image or array -> uint8 RGB array."""
    if isinstance(img, Image.Image):
        img = np.asarray(img.convert("RGB"))
    a = np.asarray(img)
    if a.ndim == 2:
        a = np.stack([a] * 3, axis=2)
    return np.ascontiguousarray(a[..., :3]).astype(np.uint8)


def luminance(a):
    """brightness of every pixel, 0 (black) to 1 (white)"""
    return (a.astype(np.float32) @ np.array([0.299, 0.587, 0.114], np.float32)) / 255.0


def noise_sigma(g):
    """Immerkaer's fast estimate of random noise (same as the loop notebook's), 0 to 1 scale"""
    k = np.array([[1, -2, 1], [-2, 4, -2], [1, -2, 1]], np.float32)
    conv = cv2.filter2D(g.astype(np.float32), -1, k, borderType=cv2.BORDER_REFLECT)[1:-1, 1:-1]
    H, W = g.shape
    return float(np.sqrt(np.pi / 2) * np.abs(conv).sum() / (6 * (W - 2) * (H - 2)))


# ---------------------------------------------------------------- fixes
def fix_brighten(a, target=0.45, max_gain=4.0):
    """multiply every pixel by the same number so the average brightness reaches `target` (never darkens)"""
    m = float(luminance(a).mean())
    gain = min(max(target / max(m, 1e-3), 1.0), max_gain)
    return np.clip(a.astype(np.float32) * gain, 0, 255).astype(np.uint8)


def fix_gamma(a, target=0.45):
    """gamma curve: lifts dark tones a lot and bright tones a little, so nothing burns out (never darkens)"""
    m = float(np.clip(luminance(a).mean(), 1e-3, 0.999))
    gamma = float(np.clip(np.log(target) / np.log(m), 0.25, 1.0))
    lut = np.clip(255.0 * (np.arange(256) / 255.0) ** gamma, 0, 255).astype(np.uint8)
    return cv2.LUT(a, lut)


def fix_clahe(a, clip=2.0, tiles=8):
    """CLAHE: evens out brightness tile by tile (8 x 8 tiles), with a cap so grain isn't blown up"""
    lab = cv2.cvtColor(a, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tiles, tiles)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def fix_contrast(a, low=1, high=99):
    """stretch the darkest 1% to black and the brightest 1% to white"""
    g = luminance(a) * 255
    lo, hi = np.percentile(g, [low, high])
    if hi - lo < 5:
        return a.copy()
    return np.clip((a.astype(np.float32) - lo) * 255.0 / (hi - lo), 0, 255).astype(np.uint8)


def fix_denoise(a):
    """non-local means: averages each small patch with similar patches elsewhere; strength follows the measured grain"""
    s = noise_sigma(luminance(a)) * 255
    h = float(np.clip(0.9 * s, 3, 30))
    return cv2.fastNlMeansDenoisingColored(a, None, h, h, 7, 21)


def fix_sharpen(a, amount=1.0, sigma=2.0):
    """unsharp mask: adds back the difference between the photo and a blurred copy, so edges look crisper"""
    blur = cv2.GaussianBlur(a, (0, 0), sigma)
    return cv2.addWeighted(a, 1 + amount, blur, -amount, 0)


def fix_smooth(a):
    """edge-preserving smoothing (mean shift): flattens grain into even patches but keeps edges,
    a bit like turning the photo into a drawing (the closest thing to 'vectorising' a photo)"""
    return cv2.pyrMeanShiftFiltering(a, sp=8, sr=24)


def fix_deblock(a):
    """gentle edge-preserving blur that hides JPEG's 8 x 8 blocks"""
    return cv2.bilateralFilter(a, 7, 40, 5)


def fix_zoom(a, keep=0.7):
    """cut out the central 70% and enlarge it back to full size"""
    H, W = a.shape[:2]
    h, w = int(round(H * keep)), int(round(W * keep))
    y, x = (H - h) // 2, (W - w) // 2
    return cv2.resize(a[y:y + h, x:x + w], (W, H), interpolation=cv2.INTER_CUBIC)


def fix_resample(a, factor=0.5):
    """shrink then enlarge: each small square of pixels is averaged, which cancels some grain"""
    H, W = a.shape[:2]
    small = cv2.resize(a, (max(8, int(W * factor)), max(8, int(H * factor))), interpolation=cv2.INTER_AREA)
    return cv2.resize(small, (W, H), interpolation=cv2.INTER_CUBIC)


def fix_white_balance(a):
    """grey world: scale red, green and blue so their averages match, removing a colour cast"""
    f = a.astype(np.float32)
    means = f.reshape(-1, 3).mean(0)
    return np.clip(f * (means.mean() / np.maximum(means, 1e-3)), 0, 255).astype(np.uint8)


def fix_saturation(a, factor=1.5):
    """make colours 50% stronger (low light fades them)"""
    hsv = cv2.cvtColor(a, cv2.COLOR_RGB2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * factor, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)


def fix_leaf_crop(a, margin=0.08):
    """zoom to where the green leaf is (instead of a fixed centre cut); falls back to a centre zoom
    when no clear leaf area is found"""
    H, W = a.shape[:2]
    small = cv2.GaussianBlur(a, (0, 0), 3).astype(np.float32) / 255.0
    exg = 2 * small[..., 1] - small[..., 0] - small[..., 2]
    mask = exg > max(0.05, float(np.percentile(exg, 60)))
    if not (0.08 < mask.mean() < 0.9):
        return fix_zoom(a)
    ys, xs = np.nonzero(mask)
    y0, y1 = np.percentile(ys, [2, 98]).astype(int)
    x0, x1 = np.percentile(xs, [2, 98]).astype(int)
    side = int(max(y1 - y0, x1 - x0) * (1 + 2 * margin))
    side = int(np.clip(side, 0.4 * min(H, W), min(H, W)))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    y = int(np.clip(cy - side // 2, 0, H - side))
    x = int(np.clip(cx - side // 2, 0, W - side))
    return cv2.resize(a[y:y + side, x:x + side], (W, H), interpolation=cv2.INTER_CUBIC)


FIXES = {
    "brighten": fix_brighten, "gamma": fix_gamma, "clahe": fix_clahe, "contrast": fix_contrast,
    "denoise": fix_denoise, "sharpen": fix_sharpen, "smooth": fix_smooth, "deblock": fix_deblock,
    "zoom": fix_zoom, "resample": fix_resample, "white_balance": fix_white_balance,
    "saturation": fix_saturation, "leaf_crop": fix_leaf_crop,
}


def apply_chain(a, steps):
    """apply fixes in order, e.g. apply_chain(photo, ["brighten", "zoom"])"""
    a = to_array(a)
    for s in steps:
        a = FIXES[s](a)
    return a


# ---------------------------------------------------------------- quality measurements
FEATURES = ["brightness", "dark_share", "bright_share", "contrast", "dynamic_range", "saturation", "colourfulness",
            "grain", "grain_relative", "colour_grain", "sharpness", "fine_detail", "edges",
            "blur_direction", "blockiness", "leaf_cover"]


def measure(img):
    """16 numbers that describe what is wrong (or right) with a photo"""
    a = to_array(img)
    if max(a.shape[:2]) != 224:
        a = cv2.resize(a, (224, 224), interpolation=cv2.INTER_AREA)
    f = a.astype(np.float32) / 255.0
    g = luminance(a)
    out = {}
    out["brightness"] = float(g.mean())
    out["dark_share"] = float((g < 0.12).mean())
    out["bright_share"] = float((g > 0.95).mean())
    out["contrast"] = float(g.std())
    p2, p98 = np.percentile(g, [2, 98])
    out["dynamic_range"] = float(p98 - p2)
    mx, mn = f.max(2), f.min(2)
    out["saturation"] = float(np.where(mx > 0, (mx - mn) / np.maximum(mx, 1e-6), 0).mean())
    rg, yb = f[..., 0] - f[..., 1], 0.5 * (f[..., 0] + f[..., 1]) - f[..., 2]
    out["colourfulness"] = float(np.hypot(rg.std(), yb.std()) + 0.3 * np.hypot(rg.mean(), yb.mean()))
    s = noise_sigma(g)
    out["grain"] = s
    out["grain_relative"] = float(s / max(g.mean(), 0.02))
    ycc = cv2.cvtColor(a, cv2.COLOR_RGB2YCrCb).astype(np.float32) / 255.0
    out["colour_grain"] = float((noise_sigma(ycc[..., 1]) + noise_sigma(ycc[..., 2])) / 2)
    lap = cv2.Laplacian(g, cv2.CV_32F)
    out["sharpness"] = float(np.log(lap.var() + 1e-8))
    P = np.abs(np.fft.fftshift(np.fft.fft2(g - g.mean()))) ** 2
    H, W = g.shape
    yy, xx = np.mgrid[-H // 2:H - H // 2, -W // 2:W - W // 2]
    r = np.hypot(yy / (H / 2), xx / (W / 2))
    band = (r > 0.02) & (r <= 1)
    out["fine_detail"] = float(P[(r > 0.35) & (r <= 1)].sum() / (P[band].sum() + 1e-12))
    u8 = (g * 255).astype(np.uint8)
    med = float(np.median(u8))
    out["edges"] = float((cv2.Canny(u8, int(max(0, 0.66 * med)), int(min(255, 1.33 * med + 1))) > 0).mean())
    # motion blur smears in one direction: compare spectral energy across 12 directions (middle frequencies)
    ang = np.mod(np.arctan2(yy, xx), np.pi)
    mid = (r > 0.08) & (r < 0.5)
    e = np.array([P[mid & (ang >= k * np.pi / 12) & (ang < (k + 1) * np.pi / 12)].mean() for k in range(12)])
    out["blur_direction"] = float(np.log((e.max() + 1e-12) / (e.min() + 1e-12)))
    dh = np.abs(np.diff(g, axis=1))
    dv = np.abs(np.diff(g, axis=0))
    bh = dh[:, 7::8].mean() / (np.delete(dh, np.s_[7::8], axis=1).mean() + 1e-6)
    bv = dv[7::8, :].mean() / (np.delete(dv, np.s_[7::8], axis=0).mean() + 1e-6)
    out["blockiness"] = float((bh + bv) / 2)
    exg = 2 * f[..., 1] - f[..., 0] - f[..., 2]
    out["leaf_cover"] = float((exg > 0.05).mean())
    return out

In [ ]:
# ---- CLIP helpers: the same model, image preparation and vote as Step 2 ----
CLIP = {}

def clip_start():
    # loads CLIP once; returns False if it can't be loaded
    if CLIP:
        return True
    try:
        import torch
        from transformers import CLIPModel, AutoTokenizer
        device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Loading {CLIP_MODEL} on " + ("the GPU" if device == "cuda" else
              "the CPU (slower; Runtime > Change runtime type > T4 GPU makes it faster)") + " ...")
        model = CLIPModel.from_pretrained(CLIP_MODEL)
        model = (model.half() if device == "cuda" else model.float()).to(device).eval()
        CLIP.update(torch=torch, model=model, tok=AutoTokenizer.from_pretrained(CLIP_MODEL), device=device,
                    dtype=next(model.parameters()).dtype, size=model.config.vision_config.image_size,
                    MEAN=torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1),
                    STD=torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1))
        return True
    except Exception as e:
        print("Couldn't load CLIP:", short(e))
        return False

def clip_stop():
    if CLIP:
        dev, torch = CLIP["device"], CLIP["torch"]
        CLIP.clear()
        if dev == "cuda":
            torch.cuda.empty_cache()

def clip_prep(a):
    # exactly Step 2's preparation: shortest side to 224 (bicubic), centre square, CLIP's colour scaling
    torch, size = CLIP["torch"], CLIP["size"]
    img = Image.fromarray(a)
    w, h = img.size
    s = size / min(w, h)
    if s != 1:
        img = img.resize((max(size, round(w * s)), max(size, round(h * s))), Image.BICUBIC)
    w, h = img.size
    left, top = (w - size) // 2, (h - size) // 2
    img = img.crop((left, top, left + size, top + size))
    x = torch.from_numpy(np.asarray(img).astype(np.float32) / 255.0).permute(2, 0, 1)
    return (x - CLIP["MEAN"]) / CLIP["STD"]

def clip_image_vectors(items, batch=32, desc="CLIP: images"):
    # items: photo file paths or RGB arrays; returns one unit-length vector per item
    torch, model = CLIP["torch"], CLIP["model"]
    out = []
    with torch.no_grad():
        for i in tqdm(range(0, len(items), batch), desc=desc):
            chunk = [x if isinstance(x, np.ndarray) else to_array(Image.open(x)) for x in items[i:i + batch]]
            x = torch.stack([clip_prep(a) for a in chunk]).to(device=CLIP["device"], dtype=CLIP["dtype"])
            v = model.visual_projection(model.vision_model(pixel_values=x).pooler_output)
            out.append(torch.nn.functional.normalize(v.float(), dim=-1).cpu())
    return torch.cat(out).numpy()

def clip_text_vectors(texts):
    torch, model = CLIP["torch"], CLIP["model"]
    with torch.no_grad():
        t = CLIP["tok"](texts, padding=True, truncation=True, max_length=77, return_tensors="pt")
        v = model.text_projection(model.text_model(input_ids=t["input_ids"].to(CLIP["device"]),
                                                   attention_mask=t["attention_mask"].to(CLIP["device"])).pooler_output)
        return torch.nn.functional.normalize(v.float(), dim=-1).cpu().numpy()

def option_sentences(l):
    # the same four sentences per option as Step 2's zero-shot
    if l["key"].endswith("_healthy"):
        c = l["crop"]
        return [f"a photo of a healthy {c} plant.", f"a photo of healthy {c} leaves.",
                f"a close-up photo of a healthy {c} leaf.", f"a photo of a {c} plant with no disease."]
    n, c = l["clip"], l["crop"]
    return [f"a photo of {n}.", f"a photo of a plant with {n}.",
            f"a close-up photo of a leaf with {n}.", f"a photo of {c} plants affected by {n}."]

def option_vectors():
    T = np.stack([clip_text_vectors(option_sentences(l)).mean(axis=0) for l in LABELS])
    return T / np.linalg.norm(T, axis=1, keepdims=True)

def database_vote(S, ref_labels, k):
    # S: similarity of each query (rows) to each reference (columns); -inf = not allowed.
    # The k most similar references vote, weighted by similarity (Step 2's rule).
    preds, confs = [], []
    k = max(1, min(k, S.shape[1]))
    nearest = np.argsort(-S, axis=1)[:, :k]
    for i in range(S.shape[0]):
        votes = {}
        for j in nearest[i]:
            if np.isfinite(S[i, j]):
                votes[ref_labels[j]] = votes.get(ref_labels[j], 0.0) + max(float(S[i, j]), 0.0) + 1e-6
        if not votes:
            preds.append(""); confs.append(0.0); continue
        best = max(votes, key=votes.get)
        preds.append(best)
        confs.append(votes[best] / sum(votes.values()))
    return preds, confs, nearest

In [ ]:
# ---- Gemini helpers: the same prompts, pacing, retries and saving as Steps 2 and 2b ----
class DailyLimit(Exception): pass
class KeyProblem(Exception): pass
class EmptyReply(Exception): pass
class NotInFreeTier(Exception): pass

GEM = {}   # filled by gemini_start(): client, types, errors
COLUMNS = ["image", "model", "job", "answer", "pred_label", "confidence", "clear_enough", "references", "seconds", "time"]

def get_api_key():
    for name in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        try:
            from google.colab import userdata
            k = userdata.get(name)
            if k:
                return k.strip()
        except Exception:
            pass
        if os.environ.get(name):
            return os.environ[name].strip()
    from getpass import getpass
    print("No Colab secret called GEMINI_API_KEY was found (see the steps at the top).")
    return getpass("Paste your Gemini API key here and press Enter (leave empty to skip Gemini): ").strip()

def gemini_start():
    if GEM:
        return True
    key = get_api_key()
    if not key:
        print("No API key, so Gemini is skipped; the tables use the answers saved so far.")
        return False
    from google import genai
    from google.genai import types, errors
    GEM.update(client=genai.Client(api_key=key), types=types, errors=errors)
    return True

def make_config(options):
    types = GEM["types"]
    return types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=types.Schema(
            type=types.Type.OBJECT,
            properties={"diagnosis": types.Schema(type=types.Type.STRING, enum=list(options)),
                        "confidence": types.Schema(type=types.Type.NUMBER),
                        "clear_enough": types.Schema(type=types.Type.BOOLEAN)},
            required=["diagnosis", "confidence", "clear_enough"]))

def norm_text(s):
    return re.sub(r"[^a-z0-9]+", " ", str(s).lower()).strip()

def key_for(answer, options):
    # the label key of Gemini's answer, matched only against this job's options
    if answer in options:
        return OPTION_TO_KEY[answer]
    norm = {norm_text(o): o for o in options}
    n = norm_text(answer)
    if n in norm:
        return OPTION_TO_KEY[norm[n]]
    hits = [o for k, o in norm.items() if k and k in n]
    return OPTION_TO_KEY[max(hits, key=len)] if hits else ""

def parse_reply(text, options):
    text = (text or "").strip()
    data = {}
    m = re.search(r"\{.*\}", text, re.S)
    if m:
        try:
            data = json.loads(m.group(0))
        except Exception:
            data = {}
    if not isinstance(data, dict):
        data = {}
    answer = data.get("diagnosis") or text
    try:
        conf = float(data.get("confidence"))
        conf = conf / 100 if 1 < conf <= 100 else conf
        conf = min(max(conf, 0.0), 1.0)
    except Exception:
        conf = ""
    clear = data.get("clear_enough", "")
    if isinstance(clear, str):
        clear = {"true": True, "yes": True, "false": False, "no": False}.get(clear.strip().lower(), "")
    return str(answer), key_for(answer, options), conf, clear

def append_row(path, row, columns):
    new = not os.path.exists(path)
    with open(path, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=columns)
        if new:
            w.writeheader()
        w.writerow({c: row.get(c, "") for c in columns})

def load_answers(path, model=None):
    # saved answers in one file, one row per image (the latest)
    empty = pd.DataFrame(columns=["image", "model", "answer", "pred_label", "confidence", "clear_enough"])
    if not os.path.exists(path):
        return empty
    c = pd.read_csv(path, dtype=str, keep_default_na=False)
    if not {"image", "model", "answer", "pred_label"} <= set(c.columns):
        print("Ignoring", os.path.basename(path), "(unexpected columns)")
        return empty
    c = c[(c.model == (model or GEMINI_MODEL)) & ((c.pred_label != "") | (c.answer != ""))]
    return c.drop_duplicates("image", keep="last")

MIME = {".png": "image/png", ".jpg": "image/jpeg", ".jpeg": "image/jpeg", ".webp": "image/webp"}

def part_from_path(path):
    with open(path, "rb") as f:
        data = f.read()
    return GEM["types"].Part.from_bytes(data=data, mime_type=MIME.get(os.path.splitext(path)[1].lower(), "image/png"))

def part_from_array(a, fmt="PNG", max_side=None):
    img = Image.fromarray(a)
    if max_side:
        img.thumbnail((max_side, max_side))
    buf = io.BytesIO()
    img.save(buf, format=fmt, **({"quality": 90} if fmt == "JPEG" else {}))
    return GEM["types"].Part.from_bytes(data=buf.getvalue(), mime_type="image/jpeg" if fmt == "JPEG" else "image/png")

def plain_prompt(options):
    # Step 2's prompt, with the given list of options
    return ("You are an expert plant pathologist. The photo shows part of a crop plant. "
            "It may be blurry, dark, grainy or compressed.\n"
            "Which ONE option below best describes the main disease, pest damage or condition you can see? "
            "Pick the single most likely option, even if you are unsure.\n\nOptions:\n"
            + "\n".join("- " + o for o in options)
            + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list), \"confidence\" "
              "(from 0 to 1) and \"clear_enough\" (true if the photo is clear enough for a reliable diagnosis, "
              "false if the farmer should take it again).")

def help_head(n_refs):
    # Step 2b's opening text
    head = ("You are an expert plant pathologist. The first image is a farmer's photo of part of a crop plant. "
            "It may be blurry, dark, grainy or compressed.\n")
    if n_refs:
        head += (f"After it come {n_refs} reference photos from a labelled collection of clean photos, each with "
                 "its correct label. A similarity search picked them as the photos that look most like the farmer's "
                 "photo, most similar first. Use them as examples to compare with: they can show a different "
                 "disease from the farmer's photo.\n")
    return head

def help_question(options, with_descriptions=True):
    # Step 2b's question, with descriptions next to the options
    lines = "\n".join(f"- {o} — {DESCRIPTIONS[OPTION_TO_KEY[o]]}" if with_descriptions and DESCRIPTIONS.get(OPTION_TO_KEY[o])
                      else f"- {o}" for o in options)
    return ("Which ONE option below best describes the main disease, pest damage or condition you can see in the "
            "farmer's photo? Pick the single most likely option, even if you are unsure.\n\nOptions"
            + (", each with what it looks like" if with_descriptions else "") + ":\n" + lines
            + "\n\nAnswer in JSON with \"diagnosis\" (exactly one option from the list"
            + (", without its description" if with_descriptions else "") + "), \"confidence\" (from 0 to 1) and "
              "\"clear_enough\" (true if the farmer's photo is clear enough for a reliable diagnosis, false if the farmer "
              "should take it again).")

_last_call = [0.0]
_rpm = [float(GEMINI_REQUESTS_PER_MINUTE)]
_busy_in_a_row, _ok_streak = [0], [0]

def pace():
    gap = 60.0 / max(_rpm[0], 0.1)
    wait = _last_call[0] + gap - time.time()
    if wait > 0:
        time.sleep(wait)
    _last_call[0] = time.time()

def ask(contents, config):
    resp = GEM["client"].models.generate_content(model=GEMINI_MODEL, contents=contents, config=config)
    try:
        text = resp.text
    except Exception:
        text = None
    if not text:
        why = ""
        try:
            why = str(resp.candidates[0].finish_reason)
        except Exception:
            pass
        raise EmptyReply(why.strip() or "empty reply")
    return text

def retry_seconds(msg):
    m = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", msg) or re.search(r"retry in (\d+(?:\.\d+)?)\s*s", msg, re.I)
    return float(m.group(1)) if m else None

def call_gemini(contents, config, tries=8):
    last = None
    for attempt in range(tries):
        pace()
        try:
            text = ask(contents, config)
            _busy_in_a_row[0] = 0
            _ok_streak[0] += 1
            if _ok_streak[0] % 20 == 0:
                _rpm[0] = min(float(GEMINI_REQUESTS_PER_MINUTE), _rpm[0] * 1.15)
            return text
        except GEM["errors"].APIError as e:
            code, msg, last = getattr(e, "code", None), str(e), e
            if code == 429:
                if re.search(r"limit:\s*0\b", msg):
                    raise NotInFreeTier(msg)
                if re.search(r"PerDay|per day|daily", msg, re.I):
                    raise DailyLimit(msg)
                _busy_in_a_row[0] += 1
                _ok_streak[0] = 0
                if _busy_in_a_row[0] >= 8:
                    raise DailyLimit(msg)
                if _rpm[0] > 2:
                    _rpm[0] = max(2.0, _rpm[0] * 0.8)
                wait = (retry_seconds(msg) or 20 * (attempt + 1)) + 2
                print(f"  too many requests; waiting {wait:.0f} s, then continuing at {_rpm[0]:.1f} per minute ...")
                time.sleep(wait)
                continue
            if code is None or code >= 500:
                wait = min(120, 5 * 2 ** attempt)
                print(f"  Gemini is busy ({code}); trying again in {wait} s ...")
                time.sleep(wait)
                continue
            if code in (401, 403) or re.search(r"API.?key", msg, re.I):
                raise KeyProblem(msg)
            raise
        except EmptyReply as e:
            last = e
            if attempt >= 1:
                raise
        except Exception as e:
            last = e
            wait = min(120, 5 * 2 ** attempt)
            print(f"  connection problem ({short(e, 80)}); trying again in {wait} s ...")
            time.sleep(wait)
    raise RuntimeError(f"gave up after {tries} tries: {short(last, 200)}")

def run_job(name, cache_path, queue, build, options, extra=None):
    """Asks Gemini about every row of `queue` without a saved answer in cache_path.
    build(row) gives the request contents; extra(row) gives extra columns to save.
    Returns 'done', 'daily', 'key' or 'stopped'."""
    done = set(load_answers(cache_path).image)
    todo = [r for r in queue if r.image not in done]
    if not todo:
        print(f"{name}: every image is answered already ({len(queue)}).")
        return "done"
    if not gemini_start():
        return "key"
    config = make_config(options)
    print(f"\n{name}: {len(todo)} images to go, about {len(todo) / _rpm[0]:.0f} minutes at {_rpm[0]:.0f} per minute. "
          "You can stop at any time; progress is saved.")
    n_ok, n_err, errors_in_a_row, status = 0, 0, 0, "done"
    bar = tqdm(total=len(todo), desc=name)
    try:
        for row in todo:
            t0 = time.time()
            try:
                text = call_gemini(build(row), config)
                answer, key, conf, clear = parse_reply(text, options)
                rec = {"image": row.image, "model": GEMINI_MODEL, "job": name, "answer": answer[:300], "pred_label": key,
                       "confidence": conf, "clear_enough": clear, "seconds": round(time.time() - t0, 1),
                       "time": datetime.datetime.now().isoformat(timespec="seconds")}
                if extra:
                    rec.update(extra(row))
                append_row(cache_path, rec, COLUMNS)
                n_ok += 1
                errors_in_a_row = 0
            except DailyLimit:
                print("\nGemini's free quota for today seems to be used up. Progress is saved: run this notebook again "
                      "after midnight Pacific time (about 2 pm in Bangladesh) and it continues where it stopped.")
                status = "daily"
                break
            except (KeyProblem, NotInFreeTier) as e:
                print("\nGemini refused the key or the model:", short(e, 200))
                status = "key"
                break
            except Exception as e:
                n_err += 1
                errors_in_a_row += 1
                append_row(os.path.join(OUT_DIR, "gemini_errors.csv"),
                           {"image": row.image, "model": GEMINI_MODEL, "job": name, "error": short(e, 500),
                            "time": datetime.datetime.now().isoformat(timespec="seconds")},
                           ["image", "model", "job", "error", "time"])
                if (n_ok == 0 and errors_in_a_row >= 2) or errors_in_a_row >= 10:
                    print("\nGemini keeps failing, so it stops for now. Last error:", short(e, 300))
                    status = "stopped"
                    break
            bar.update(1)
    except KeyboardInterrupt:
        print("\nStopped. Answers so far are saved; run this cell again to continue.")
        status = "stopped"
    bar.close()
    total = len(set(load_answers(cache_path).image) & {r.image for r in queue})
    print(f"{name}: {n_ok} answers this run, {n_err} errors. {total} of {len(queue)} images answered in total.")
    return status

def gemini_queue(levels, photos_per_disease=None, skip_photos=()):
    # a fixed random order of photos (balanced partial runs), each with the chosen levels
    photos = meta.drop_duplicates("photo_id")[["photo_id", "disease"]]
    rng = random.Random(SEED)
    keep = []
    for d, ids in photos.groupby("disease").photo_id:
        ids = sorted(i for i in ids if i not in set(skip_photos))
        rng.shuffle(ids)
        keep += ids if photos_per_disease is None else ids[:photos_per_disease]
    keep = sorted(keep)
    random.Random(SEED + 1).shuffle(keep)
    by_photo = {pid: g for pid, g in meta.groupby("photo_id")}
    return [r for pid in keep for r in by_photo[pid].sort_values("level").itertuples(index=False) if r.level in levels]

## 3. Extra reference photos (options 2 and 4)

**From PlantVillage (automatic):** 30 photos each of cedar apple rust, healthy apple, potato early blight, healthy potato, grape black rot, grape esca and healthy grape. These are look-alikes of our diseases and healthy leaves of the same crops. Also 30 PlantVillage apple scab photos, used **only in option 4** as apple scab from another dataset. Potato late blight and grape leaf blight are never downloaded, because our own test photos of those two came from PlantVillage and could be the very same photos.

**Your own photos (optional):** put them in your Drive like this. The folder name must be the option's key from the table in section 2 (for example `rice_blast`, `jute_healthy`, `pear_rust`). A subfolder can name the dataset the photos came from:

```
My Drive/SafeCrop/extra_references/
    rice_blast/
        Mendeley_rice_leaf/      <- the dataset's name (optional)
            photo1.jpg ...
    jute_healthy/
        photo1.jpg ...           <- no subfolder: the source is recorded as "your photos"
    potato_late_blight/
        PLD_field/               <- one of OUR diseases from ANOTHER dataset: used only in option 4
            photo1.jpg ...
```

Good free sources: **AppleLeaf9** (apple Alternaria leaf spot, powdery mildew, rust and healthy), **DiaMOS Plant** (healthy pear), the **Blackgram Plant Leaf Disease** dataset on Mendeley (black gram yellow mosaic, powdery mildew, healthy), and rice leaf disease datasets on Mendeley or UCI (blast, brown spot, bacterial leaf blight).

Every extra photo is cut to the central square and resized to 224 × 224, exactly like our images. A photo that is a copy of one of our 210 photos (same picture fingerprint, or almost the same CLIP vector) is dropped, so no test photo can find itself. We checked this: all 25 of our potato late blight photos and all 30 grape leaf blight photos are PlantVillage photos, which is why those two folders are never downloaded. None of our 30 apple scab photos (AppleLeaf9) is in PlantVillage's apple scab folder.

In [ ]:
EXTRA_DIR = os.path.join(WORK, "extra_references")    # every extra photo used by options 2 and 4
os.makedirs(EXTRA_DIR, exist_ok=True)
# PlantVillage folder for each option it has (never potato late blight or grape leaf blight: our test photos come from there)
PLANTVILLAGE = {
    "apple_cedar_rust": "Apple___Cedar_apple_rust", "apple_healthy": "Apple___healthy",
    "potato_early_blight": "Potato___Early_blight", "potato_healthy": "Potato___healthy",
    "grape_black_rot": "Grape___Black_rot", "grape_esca": "Grape___Esca_(Black_Measles)",
    "grape_healthy": "Grape___healthy",
    "apple_scab": "Apple___Apple_scab",   # apple scab from another dataset: option 4 only
}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

def fetch_plantvillage():
    repo = os.path.join(WORK, "plantvillage_repo")
    if not os.path.isdir(os.path.join(repo, ".git")):
        print("Getting the PlantVillage file list from GitHub (file names only, a few seconds) ...")
        subprocess.run(["git", "clone", "-q", "--filter=blob:none", "--no-checkout", "--depth", "1",
                        "https://github.com/spMohanty/PlantVillage-Dataset", repo], check=True, timeout=900)
    picked = {}
    for key, folder in PLANTVILLAGE.items():
        names = subprocess.run(["git", "-C", repo, "ls-tree", "--name-only", "HEAD", f"raw/color/{folder}/"],
                               capture_output=True, text=True, check=True).stdout.splitlines()
        names = sorted(n for n in names if n.lower().endswith(IMG_EXT))
        random.Random(f"{SEED}/{key}").shuffle(names)
        picked[key] = names[:EXTRA_PER_OPTION]
    wanted = [p for ps in picked.values() for p in ps]
    print(f"Downloading {len(wanted)} PlantVillage photos ...")
    # check out exactly these files (git 2.26 or newer; Colab has it), fetching only their contents
    subprocess.run(["git", "--literal-pathspecs", "-C", repo, "checkout", "-q", "HEAD",
                    "--pathspec-from-file=-", "--pathspec-file-nul"],
                   input="\0".join(wanted) + "\0", text=True, check=True, timeout=1800)
    n = 0
    for key, ps in picked.items():
        dest = os.path.join(EXTRA_DIR, key, "PlantVillage")
        os.makedirs(dest, exist_ok=True)
        for p in ps:
            src = os.path.join(repo, p)
            if os.path.exists(src):
                shutil.copy(src, os.path.join(dest, os.path.basename(p)))
                n += 1
    print(f"  {n} PlantVillage photos ready.")

def collect(top, origin):
    # option folder -> (optional source subfolder) -> photos
    rows = []
    if not top or not os.path.isdir(top):
        return rows
    for key in sorted(os.listdir(top)):
        folder = os.path.join(top, key)
        if not os.path.isdir(folder):
            continue
        if key not in LABEL_KEYS:
            print(f"  Skipping folder '{key}': not one of the 33 option keys (see section 2).")
            continue
        for root, _, files in os.walk(folder):
            rel = os.path.relpath(root, folder)
            source = "your photos" if rel == "." else rel.split(os.sep)[0]
            for f in sorted(files):
                if f.lower().endswith(IMG_EXT) and not f.startswith("._"):
                    rows.append(dict(path=os.path.join(root, f), option=key, source=source, origin=origin))
    return rows

def load_square(path, size=224):
    # exactly how our images were made: upright, central square, 224 x 224 (bicubic)
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return np.asarray(img.crop((left, top, left + s, top + s)).resize((size, size), Image.BICUBIC))

if DOWNLOAD_PLANTVILLAGE:
    try:
        fetch_plantvillage()
    except Exception as e:
        print("Couldn't download from PlantVillage:", short(e, 200), "- carrying on with your own photos only.")
rows = collect(EXTRA_DIR, "downloaded")
user_dir = os.path.join("/content/drive/MyDrive", EXTRA_FOLDER) if DRIVE_OK else None
if user_dir and os.path.isdir(user_dir):
    rows += collect(user_dir, "your Drive folder")
elif DRIVE_OK:
    print(f"No folder My Drive/{EXTRA_FOLDER} yet; only the downloaded photos are used.")
extra = pd.DataFrame(rows, columns=["path", "option", "source", "origin"])
# at most EXTRA_PER_OPTION photos per option and source, picked at random (fixed seed)
extra = extra.drop_duplicates("path")
parts = [g.sample(min(len(g), EXTRA_PER_OPTION), random_state=SEED) for _, g in extra.groupby(["option", "source"])]
extra = pd.concat(parts, ignore_index=True) if parts else extra.reset_index(drop=True)
ok = []
for p in extra.path:
    try:
        Image.open(p).verify()
        ok.append(True)
    except Exception:
        ok.append(False)
if not all(ok):
    print(f"  {len(ok) - sum(ok)} extra files aren't readable photos and are skipped.")
extra = extra[ok].reset_index(drop=True)
extra["target"] = extra.option.isin(TARGETS)    # one of our 7 diseases (from another dataset): option 4 only
print(f"\n{len(extra)} extra photos for {extra.option.nunique()} options.")
if len(extra):
    display(extra.groupby(["option", "source", "origin"]).size().rename("photos").to_frame())

## 4. CLIP features for every photo

CLIP turns each of our 840 images, each extra photo and each of the 33 options into a vector, exactly as in Step 2. The vectors are saved, so a second run doesn't need CLIP again. Extra photos that are near-copies of our photos are dropped here.

In [ ]:
CLIP_CACHE = os.path.join(OUT_DIR, "clip_features.npz")
V = X = T = None
if os.path.exists(CLIP_CACHE):
    try:
        z = np.load(CLIP_CACHE, allow_pickle=False)
        if (str(z["model"]) == CLIP_MODEL and list(z["images"]) == list(meta.image)
                and [str(p) for p in z["extra"]] == list(extra.path)):
            V, X, T = z["V"], z["X"], z["T"]
            print("CLIP features are already saved, so CLIP isn't loaded again.")
    except Exception as e:
        print("The saved CLIP features couldn't be used (" + short(e, 80) + "), so they are made again.")
if V is None:
    if not clip_start():
        raise RuntimeError("CLIP couldn't be loaded. Check the internet connection and run the notebook again.")
    V = clip_image_vectors(list(meta.path), desc="CLIP: our 840 images")
    X = (clip_image_vectors([load_square(p) for p in extra.path], desc="CLIP: extra photos") if len(extra)
         else np.zeros((0, V.shape[1]), np.float32))
    T = option_vectors()
    np.savez(CLIP_CACHE, model=np.array(CLIP_MODEL), images=np.array(list(meta.image)),
             extra=np.array(list(extra.path), dtype=str), V=V, X=X, T=T)
    clip_stop()

L0 = np.flatnonzero((meta.level == "L0").to_numpy())     # our clean photos: the database in Step 2
PATH_OF = dict(zip(meta.image, meta.path))
REF_PID, REF_DIS = meta.photo_id.to_numpy()[L0], meta.disease.to_numpy()[L0]
QPID = meta.photo_id.to_numpy()

# ---- drop extra photos that are near-copies of our photos ----
def fingerprint(a, n=16):
    # 256 yes/no answers to "is this spot brighter than the one to its right?" - copies give (almost) the same answers
    g = np.asarray(Image.fromarray(a).convert("L").resize((n + 1, n), Image.LANCZOS), dtype=np.float32)
    return (g[:, 1:] > g[:, :-1]).ravel()

if len(extra):
    sim = X @ V[L0].T
    extra["closest_of_ours"] = sim.max(1).round(3)
    extra["closest_photo"] = REF_PID[sim.argmax(1)]
    ours_fp = np.array([fingerprint(to_array(Image.open(PATH_OF[meta.image.iat[i]]))) for i in L0])
    extra_fp = np.array([fingerprint(load_square(p)) for p in extra.path])
    extra["fingerprint_difference"] = (extra_fp[:, None, :] != ours_fp[None, :, :]).sum(2).min(1)
    dup = ((extra.closest_of_ours > DUPLICATE_LIMIT) | (extra.fingerprint_difference <= 20)).to_numpy()
    if dup.any():
        print(f"{dup.sum()} extra photos are near-copies of our photos and are dropped:")
        display(extra[dup].groupby(["option", "source"]).size().rename("dropped").to_frame())
    else:
        print("No extra photo is a near-copy of our photos.")
    extra.to_csv(os.path.join(OUT_DIR, "extra_photos.csv"), index=False)
    keep = ~dup
    extra, X = extra[keep].reset_index(drop=True), X[keep]
print(f"{len(extra)} extra photos are used.")

# ---- everything the options predict goes into PRED ----
PRED = []
LEVELS = ["L0", "L1", "L2", "L3"]

def add(option, condition, family, choices, images, preds, confs=None, draw=0):
    images, preds = list(images), list(preds)
    PRED.append(pd.DataFrame(dict(option=option, condition=condition, family=family, choices=choices, image=images,
                                  pred_label=preds, confidence=list(confs) if confs is not None else np.nan, draw=draw)))

def scored():
    P = pd.concat(PRED, ignore_index=True)
    P = P.merge(meta[["image", "photo_id", "disease", "group", "level", "noise", "source"]], on="image")
    P["correct"] = P.pred_label == P.disease
    return P

def level_table(P, conditions=None):
    if conditions is not None:
        P = P[P.condition.isin(conditions)]
    by_draw = P.groupby(["condition", "draw", "level"]).correct.mean()
    t = by_draw.groupby(["condition", "level"]).mean().mul(100).unstack("level").reindex(columns=LEVELS)
    t["All levels"] = P.groupby(["condition", "draw"]).correct.mean().groupby("condition").mean().mul(100)
    t["Images"] = P.groupby(["condition", "draw"]).image.nunique().groupby("condition").mean().round().astype(int)
    order = list(dict.fromkeys(P.condition))
    return t.reindex(order)

## 5. Option 1: same options for everyone

Every model chooses from the **same list**: first our 7 diseases, then all 33 options. Zero-shot CLIP is simply asked again with only 7 options. The database can only answer one of our 7 diseases (it has no photos of the other 26 options), so its 33-option version is option 2. Gemini with 7 options is a new Gemini job (`"7_options"` in Settings); its answers are added in section 9.

In [ ]:
LOGITS = V @ T.T
IDX7 = [LABEL_KEYS.index(k) for k in TARGETS]
zs33 = [LABEL_KEYS[i] for i in LOGITS.argmax(1)]
zs7 = [TARGETS[i] for i in LOGITS[:, IDX7].argmax(1)]

S7 = V @ V[L0].T
S7[QPID[:, None] == REF_PID[None, :]] = -np.inf          # a photo never finds its own clean version
db7, db7_conf, NN7 = database_vote(S7, REF_DIS, CLIP_DATABASE_NEIGHBOURS)

add("1", "CLIP zero-shot, 33 options (Step 2)", "CLIP zero-shot", 33, meta.image, zs33)
add("1", "CLIP zero-shot, 7 options", "CLIP zero-shot", 7, meta.image, zs7)
add("1", "CLIP database, 7 diseases (Step 2)", "CLIP database", 7, meta.image, db7, db7_conf)
plain = load_answers(GEMINI_CACHE)
if len(plain):
    add("1", "Gemini, 33 options (Step 2)", "Gemini", 33, plain.image, plain.pred_label)
else:
    print("Step 2's Gemini answers weren't found in your Drive, so plain Gemini isn't in the tables.")

P = scored()
print("Option 1, accuracy (%) by noise level:")
display(level_table(P[P.option == "1"]).round(1))

## 6. Option 2: the database filled to 33 options

The database now holds our 210 clean photos **plus** the extra photos of look-alike diseases and healthy leaves, so a noisy photo can also be matched to *"Grape: black rot"* or *"Potato: healthy"*, as zero-shot CLIP and Gemini can. Everything else is unchanged: the 5 most similar photos vote, and a photo never finds its own clean version. The table also shows how many of the 33 options have photos so far, and how often the database now answers one of the new options.

In [ ]:
ex2 = extra[~extra.target].reset_index(drop=True)          # look-alike and healthy photos only
X2 = X[~extra.target.to_numpy()]
n_options = len(set(TARGETS) | set(ex2.option))
REF33_LABELS = np.concatenate([REF_DIS, ex2.option.to_numpy()])
REF33_ID = [("ours", meta.image.iat[i]) for i in L0] + [("extra", p) for p in ex2.path]
REFS33 = {}
if len(ex2):
    S33 = np.hstack([S7, V @ X2.T])
    db33, db33_conf, NN33 = database_vote(S33, REF33_LABELS, CLIP_DATABASE_NEIGHBOURS)
    REFS33 = {meta.image.iat[i]: [(REF33_ID[j], REF33_LABELS[j]) for j in NN33[i]] for i in range(len(meta))}
    name33 = f"CLIP database, filled to {n_options} options"
    add("2", name33, "CLIP database", n_options, meta.image, db33, db33_conf)
    missing = [k for k in LABEL_KEYS if k not in set(TARGETS) | set(ex2.option)]
    print(f"The database now has photos for {n_options} of the 33 options. Still without photos: {', '.join(missing) or 'none'}.")
    P = scored()
    print("\nOption 2, accuracy (%) by noise level:")
    display(level_table(P, ["CLIP database, 7 diseases (Step 2)", name33]).round(1))
    new_answers = P[(P.condition == name33) & ~P.pred_label.isin(TARGETS)]
    share = len(new_answers) / max(1, (P.condition == name33).sum()) * 100
    print(f"The filled database answered one of the new options for {share:.0f}% of images "
          f"(those answers are always wrong: every test photo shows one of our diseases).")
    if len(new_answers):
        display(new_answers.pred_label.map(KEY_TO_OPTION).value_counts().head(10).rename("answers").to_frame())
else:
    print("No look-alike or healthy photos are available, so option 2 is skipped. "
          "Turn on DOWNLOAD_PLANTVILLAGE or add photos to your Drive folder (section 3).")

## 7. Option 3: fewer examples

The database keeps the same vote but gets only **1, 3, 5 or 10 clean photos per disease** instead of about 29. Each size is drawn **10 times** at random and the results are averaged, so one lucky or unlucky pick doesn't decide it. The photos chosen as examples are left out of the test, so no photo is ever tested against itself.

In [ ]:
PHOTOS_BY_DIS = {d: sorted(meta[meta.disease == d].photo_id.unique()) for d in TARGETS}
L0_ROW = {meta.photo_id.iat[i]: i for i in L0}
SUPPORT = {}
for k in SHOTS:
    for draw in range(DRAWS):
        rng = np.random.default_rng([SEED, k, draw])
        support = [str(p) for d in TARGETS
                   for p in rng.choice(PHOTOS_BY_DIS[d], size=min(k, len(PHOTOS_BY_DIS[d]) - 1), replace=False)]
        SUPPORT[(k, draw)] = support
        ridx = np.array([L0_ROW[p] for p in support])
        q = ~meta.photo_id.isin(support).to_numpy()
        preds, confs, _ = database_vote(V[q] @ V[ridx].T, meta.disease.to_numpy()[ridx], CLIP_DATABASE_NEIGHBOURS)
        add("3", f"CLIP database, {k} photo{'s' if k > 1 else ''} per disease", "CLIP database", 7,
            meta.image[q], preds, confs, draw=draw)

P = scored()
o3 = P[P.option == "3"]
acc3 = o3.groupby(["condition", "draw", "level"]).correct.mean().mul(100).unstack("level").reindex(columns=LEVELS)
mean3 = acc3.groupby("condition").mean()
sd3 = acc3.groupby("condition").std()
names3 = [f"CLIP database, {k} photo{'s' if k > 1 else ''} per disease" for k in SHOTS]
show3 = pd.DataFrame({lv: [f"{mean3.loc[n, lv]:.0f} ± {sd3.loc[n, lv]:.0f}" for n in names3] for lv in LEVELS}, index=names3)
step2 = level_table(P, ["CLIP database, 7 diseases (Step 2)"])
show3.loc["CLIP database, all ~29 photos per disease (Step 2)"] = [f"{step2.loc['CLIP database, 7 diseases (Step 2)', lv]:.0f}" for lv in LEVELS]
print(f"Option 3, accuracy (%) by noise level: average ± spread over {DRAWS} random picks")
display(show3)

## 8. Option 4: examples from another source

For each test photo, the database may **not use any photo from the test photo's own dataset**. If the database only wins because a PlantVillage photo looks like other PlantVillage photos (same grey background, same camera), its accuracy falls here.

This needs photos of **our** diseases from **another** dataset. The notebook downloads PlantVillage apple scab for this (our apple scab photos come from AppleLeaf9). For other diseases, add photos to your Drive folder (section 3), for example `potato_late_blight/PLD_field/`. Only diseases with at least `OTHER_SOURCE_MIN` such photos are tested, and they are compared on the same images with the database that may use any source.

In [ ]:
REF_SRC = np.concatenate([meta.source.to_numpy()[L0], extra.source.to_numpy()])
REF4_LABELS = np.concatenate([REF_DIS, extra.option.to_numpy()])
REF4_PID = np.concatenate([REF_PID, np.array(["extra"] * len(extra))])
other = {d: int(((REF4_LABELS == d) & (REF_SRC != OUR_SOURCE.get(d, ""))).sum()) for d in TARGETS}
eligible = [d for d, n in other.items() if n >= OTHER_SOURCE_MIN]
print("Photos of each disease from another dataset:", {d: n for d, n in other.items() if n} or "none")
if not eligible:
    print("No disease has enough photos from another dataset yet, so option 4 is skipped. "
          "Add some to your Drive folder (section 3), or turn on DOWNLOAD_PLANTVILLAGE.")
else:
    q = np.flatnonzero(meta.disease.isin(eligible).to_numpy())
    S4 = np.hstack([V[q] @ V[L0].T, V[q] @ X.T]) if len(extra) else V[q] @ V[L0].T
    S4[QPID[q][:, None] == REF4_PID[None, :]] = -np.inf          # never the photo's own clean version
    any_pred, any_conf, _ = database_vote(S4.copy(), REF4_LABELS, CLIP_DATABASE_NEIGHBOURS)
    S4[meta.source.to_numpy()[q][:, None] == REF_SRC[None, :]] = -np.inf   # nothing from the test photo's dataset
    oth_pred, oth_conf, _ = database_vote(S4, REF4_LABELS, CLIP_DATABASE_NEIGHBOURS)
    add("4", "CLIP database, examples from any dataset", "CLIP database", len(set(REF4_LABELS)),
        meta.image.iloc[q], any_pred, any_conf)
    add("4", "CLIP database, examples from other datasets only", "CLIP database", len(set(REF4_LABELS)),
        meta.image.iloc[q], oth_pred, oth_conf)
    P = scored()
    o4 = P[P.option == "4"]
    print(f"\nOption 4 on {', '.join(eligible)} ({len(q)} images), accuracy (%) by noise level:")
    display(level_table(o4).round(1))
    print("By disease:")
    display(o4.groupby(["disease", "condition"]).correct.mean().mul(100).unstack("condition").round(1))

## 9. Gemini jobs (they use the free daily quota)

Each job in `GEMINI_JOBS` asks Gemini about the L3 images (210 requests). Answers are saved after every image, so a job stopped by the daily limit continues on the next run.

| Job | What Gemini gets | Compared with |
|---|---|---|
| `"7_options"` | the photo and only our 7 diseases (names only, Step 2's prompt) | Gemini with 33 options (Step 2), the database |
| `"references_33"` | the photo, the **5 photos the filled 33-option database voted with** (option 2), and the 33 options with descriptions | the filled database, and Gemini with the 7-disease database's photos (Step 2b) |
| `"dictionary_1"` | the photo, **one** clean labelled photo of each of our 7 diseases (a picture dictionary), and the 33 options with descriptions | the database with 1 photo per disease (option 3) |

In [ ]:
JOB_FILES = {j: os.path.join(OUT_DIR, f"gemini_{j}.csv") for j in ["7_options", "references_33", "dictionary_1"]}
TARGET_OPTIONS = [KEY_TO_OPTION[k] for k in TARGETS]
DICT_PHOTOS = SUPPORT.get((1, 0)) or [PHOTOS_BY_DIS[d][0] for d in TARGETS]   # option 3's first 1-photo pick
_ref_cache = {}

def ref_part(ref_id):
    kind, ident = ref_id
    if ref_id not in _ref_cache:
        _ref_cache[ref_id] = to_array(Image.open(PATH_OF[ident])) if kind == "ours" else load_square(ident)
    return part_from_array(_ref_cache[ref_id], fmt="JPEG", max_side=768)

def build_7(row):
    return [plain_prompt(TARGET_OPTIONS), part_from_path(row.path)]

def build_refs33(row):
    refs = REFS33.get(row.image, [])
    parts = [help_head(len(refs)), "Farmer's photo:", part_from_path(row.path)]
    for r, (ref_id, lab) in enumerate(refs, 1):
        parts += [f"Reference photo {r}{' (most similar)' if r == 1 else ''}. Label: {KEY_TO_OPTION[lab]}", ref_part(ref_id)]
    return parts + [help_question(OPTIONS, True)]

def extra_refs33(row):
    return {"references": "; ".join(f"{i[1]} = {lab}" for i, lab in REFS33.get(row.image, []))}

DICT_REFS = [(("ours", meta[(meta.photo_id == p) & (meta.level == "L0")].image.iat[0]), meta[meta.photo_id == p].disease.iat[0])
             for p in DICT_PHOTOS]

def build_dict(row):
    head = ("You are an expert plant pathologist. The first image is a farmer's photo of part of a crop plant. "
            "It may be blurry, dark, grainy or compressed.\n"
            f"After it come {len(DICT_REFS)} reference photos: one clean, labelled photo of each disease in our collection. "
            "Use them as examples to compare with.\n")
    parts = [head, "Farmer's photo:", part_from_path(row.path)]
    for r, (ref_id, lab) in enumerate(DICT_REFS, 1):
        parts += [f"Reference photo {r}. Label: {KEY_TO_OPTION[lab]}", ref_part(ref_id)]
    return parts + [help_question(OPTIONS, True)]

JOBS = {
    "7_options": dict(options=TARGET_OPTIONS, build=build_7, extra=None, skip=(),
                      label=("1", "Gemini, 7 options", 7)),
    "references_33": dict(options=OPTIONS, build=build_refs33, extra=extra_refs33, skip=(),
                          label=("2", "Gemini + photos from the filled database + descriptions", 33)),
    "dictionary_1": dict(options=OPTIONS, build=build_dict, extra=None, skip=tuple(DICT_PHOTOS),
                         label=("3", "Gemini + 1 photo per disease + descriptions", 33)),
}
for j in GEMINI_JOBS:
    if j == "references_33" and not REFS33:
        print("Skipping references_33: the filled database (option 2) has no extra photos.")
        continue
    queue = gemini_queue(GEMINI_LEVELS, GEMINI_PHOTOS_PER_DISEASE, skip_photos=JOBS[j]["skip"])
    status = run_job(j, JOB_FILES[j], queue, JOBS[j]["build"], JOBS[j]["options"], JOBS[j]["extra"])
    if status in ("daily", "key"):
        break

for j, path in JOB_FILES.items():   # every saved answer counts, also from earlier runs
    a = load_answers(path)
    a = a[a.image.isin(set(meta.image))]
    if len(a):
        opt, name, choices = JOBS[j]["label"]
        add(opt, name, "Gemini", choices, a.image, a.pred_label)
        print(f"{name}: {len(a)} answers saved.")

## 10. Option 5 and the comparison of all options

Option 5 (Gemini with the same reference photos as the database, plus descriptions) was run in Step 2b; its answers are read from your Drive.

**How to read the tables:**
* **Accuracy by level:** each row is tested on the images in its "Images" column (Gemini jobs usually only L3).
* **Head to head:** two predictors on exactly the same images. *Fixed* = images only the second one got right, *broken* = images only the first got right. *p* below 0.05 means the difference is very unlikely to be chance (McNemar's test).

In [ ]:
from scipy.stats import binomtest
helped = load_answers(HELP_CACHE)
helped = helped[helped.image.isin(set(meta.image))]
if len(helped):
    add("5", "Gemini + photos from the 7-disease database + descriptions (Step 2b)", "Gemini", 33,
        helped.image, helped.pred_label)
else:
    print("Step 2b's answers weren't found in your Drive (SafeCrop/results/gemini_help), so option 5 isn't in the tables.")

P = scored()
P.to_csv(os.path.join(OUT_DIR, "all_predictions.csv"), index=False)
TABLE = level_table(P)
TABLE.insert(0, "Option", P.groupby("condition").option.first().reindex(TABLE.index))
TABLE.insert(1, "Choices", P.groupby("condition").choices.first().reindex(TABLE.index))
print("Accuracy (%) by noise level, every option:")
display(TABLE.round(1))

def head_to_head(a, b, level=None):
    x = P[(P.condition == a) & (P.draw == 0)][["image", "level", "correct"]]
    y = P[(P.condition == b) & (P.draw == 0)][["image", "correct"]]
    m = x.merge(y, on="image", suffixes=("_a", "_b"))
    if level:
        m = m[m.level == level]
    if not len(m):
        return None
    fixed = int((~m.correct_a & m.correct_b).sum())
    broken = int((m.correct_a & ~m.correct_b).sum())
    p = binomtest(fixed, fixed + broken).pvalue if fixed + broken else 1.0
    return dict(first=a, second=b, level=level or "all", images=len(m), first_acc=m.correct_a.mean() * 100,
                second_acc=m.correct_b.mean() * 100, change=(m.correct_b.mean() - m.correct_a.mean()) * 100,
                fixed=fixed, broken=broken, p=round(p, 4))

PAIRS = [
    ("CLIP zero-shot, 33 options (Step 2)", "CLIP zero-shot, 7 options"),
    ("Gemini, 33 options (Step 2)", "Gemini, 7 options"),
    ("CLIP database, 7 diseases (Step 2)", next((c for c in TABLE.index if c.startswith("CLIP database, filled")), "")),
    ("CLIP database, examples from any dataset", "CLIP database, examples from other datasets only"),
    ("CLIP database, 7 diseases (Step 2)", "Gemini + photos from the 7-disease database + descriptions (Step 2b)"),
    (next((c for c in TABLE.index if c.startswith("CLIP database, filled")), ""), "Gemini + photos from the filled database + descriptions"),
    ("Gemini + photos from the 7-disease database + descriptions (Step 2b)", "Gemini + photos from the filled database + descriptions"),
    ("CLIP database, 1 photo per disease", "Gemini + 1 photo per disease + descriptions"),
]
rows = []
for a, b in PAIRS:
    if a in TABLE.index and b in TABLE.index:
        r3, ra = head_to_head(a, b, "L3"), head_to_head(a, b)
        rows += [r for r in [r3, ra] if r and not (r is ra and r3 and ra["images"] == r3["images"])]
H2H = pd.DataFrame(rows)
if len(H2H):
    print("\nHead to head on the same images (change = second minus first, in points):")
    display(H2H.round({"first_acc": 1, "second_acc": 1, "change": 1, "p": 3}))

# ---------- charts ----------
FAMILY_COLOUR = {"CLIP zero-shot": "#2a78d6", "CLIP database": "#eb6834", "Gemini": "#1baf7a"}
FIG = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"font.family": "DejaVu Sans", "axes.edgecolor": "#c3c2b7", "xtick.color": "#6b7280",
                     "ytick.color": "#1f2937"})
t3 = TABLE.dropna(subset=["L3"])
fam = P.groupby("condition").family.first()
fig, ax = plt.subplots(figsize=(9.5, 0.42 * len(t3) + 1.2), dpi=160)
y = np.arange(len(t3))[::-1]
ax.barh(y, t3["L3"], color=[FAMILY_COLOUR[fam[c]] for c in t3.index], height=0.62)
for yi, (c, v) in zip(y, t3["L3"].items()):
    ax.text(v + 1, yi, f"{v:.0f}%", va="center", fontsize=8.5, color="#1f2937")
ax.set_yticks(y, [f"{t3.loc[c, 'Option']}. {c}" for c in t3.index], fontsize=8.5)
ax.set_xlim(0, 105)
ax.set_xlabel("Accuracy on severe-noise (L3) images (%)")
ax.grid(axis="x", color="#e1e0d9", lw=0.7)
ax.set_axisbelow(True)
for s in ["top", "right", "left"]:
    ax.spines[s].set_visible(False)
handles = [matplotlib.patches.Patch(color=v, label=k) for k, v in FAMILY_COLOUR.items()]
ax.legend(handles=handles, frameon=False, fontsize=8.5, loc="lower right")
ax.set_title("Fairness options: accuracy at L3, every option", loc="left", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(FIG, "fairness_L3.png"), facecolor="white")
plt.close(fig)

if len(mean3):
    xs = SHOTS + [round(np.mean([len(v) for v in PHOTOS_BY_DIS.values()]) - 1)]
    fig, ax = plt.subplots(figsize=(6.5, 4), dpi=160)
    for lv, colr in zip(LEVELS, ["#1f2937", "#2a78d6", "#eb6834", "#c0392b"]):
        ys = [mean3.loc[n, lv] for n in names3] + [step2.loc["CLIP database, 7 diseases (Step 2)", lv]]
        es = [sd3.loc[n, lv] for n in names3] + [0]
        ax.errorbar(xs, ys, yerr=es, color=colr, marker="o", ms=5, capsize=3, lw=1.8, label=lv)
    ax.set_xscale("log")
    ax.set_xticks(xs, [str(x) for x in xs])
    ax.set_xlabel("Clean photos per disease in the database")
    ax.set_ylabel("Accuracy (%)")
    ax.set_ylim(0, 100)
    ax.grid(color="#e1e0d9", lw=0.7)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.legend(title="Noise level", frameon=False, fontsize=8.5)
    ax.set_title("Option 3: how many examples the database needs", loc="left", fontsize=11)
    fig.tight_layout()
    fig.savefig(os.path.join(FIG, "option3_examples.png"), facecolor="white")
    plt.close(fig)

for f in ["fairness_L3.png", "option3_examples.png"]:
    if os.path.exists(os.path.join(FIG, f)):
        display(ShowImage(filename=os.path.join(FIG, f), width=850))

# ---------- the main findings in words ----------
def acc(c, lv="L3"):
    return TABLE.loc[c, lv] if c in TABLE.index and pd.notna(TABLE.loc[c, lv]) else None
lines = []
for a, b, what in [("CLIP zero-shot, 33 options (Step 2)", "CLIP zero-shot, 7 options", "Zero-shot CLIP with 7 instead of 33 options"),
                   ("Gemini, 33 options (Step 2)", "Gemini, 7 options", "Gemini with 7 instead of 33 options"),
                   ("CLIP database, 7 diseases (Step 2)", PAIRS[2][1], PAIRS[2][1].replace("CLIP database, filled", "The database filled") or "-"),
                   ("CLIP database, 7 diseases (Step 2)", "CLIP database, 1 photo per disease", "The database with 1 photo per disease"),
                   ("CLIP database, examples from any dataset", "CLIP database, examples from other datasets only",
                    "The database without photos from the test photo's own dataset")]:
    if acc(a) is not None and acc(b) is not None:
        lines.append(f"{what}: {acc(a):.0f}% → {acc(b):.0f}% at L3 ({acc(b) - acc(a):+.0f} points).")
print("\nIn short (L3):")
for l in lines:
    print(" •", l)

## 11. Save and download

Everything goes to `SafeCrop/results/fairness`: `fairness_tables.xlsx` (every table), `all_predictions.csv`, the extra photo list and the two charts. The zip below has the same files.

In [ ]:
xlsx = os.path.join(OUT_DIR, "fairness_tables.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    pd.DataFrame({"summary": lines + [f"CLIP model: {CLIP_MODEL}", f"Gemini model: {GEMINI_MODEL}",
                                      f"Extra photos used: {len(extra)}"]}).to_excel(w, sheet_name="Summary", index=False)
    TABLE.round(1).to_excel(w, sheet_name="All options by level")
    if len(H2H):
        H2H.round(2).to_excel(w, sheet_name="Head to head", index=False)
    show3.to_excel(w, sheet_name="Option 3 (mean ± spread)")
    (P.groupby(["condition", "draw", "group"]).correct.mean().groupby(["condition", "group"]).mean().mul(100)
       .unstack("group").round(1).to_excel(w, sheet_name="Info-rich vs scarce"))
    (P.groupby(["condition", "draw", "disease"]).correct.mean().groupby(["condition", "disease"]).mean().mul(100)
       .unstack("disease").round(1).to_excel(w, sheet_name="By disease"))
    if len(extra):
        extra.drop(columns=["path"]).to_excel(w, sheet_name="Extra photos", index=False)
print("Saved", xlsx)
zip_path = shutil.make_archive(os.path.join(WORK, "SafeCrop_fairness_results"), "zip", OUT_DIR)
print("Zip ready:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Automatic download didn't start (" + short(e, 80) + "). Use the folder icon on the left to download it.")